In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
np.random.seed(42)
n_rows=20000
genders = ['Male', 'Female', 'Non-Binary']
arrival_modes = ['Ambulance', 'Walk-in', 'Helicopter', 'Private Vehicle']
complaints = ['Chest Pain', 'Fever', 'Fracture', 'Breathing Issues', 'Abdominal Pain', 'Minor Injury']
insurance = ['Standard Health', 'Premium Care', 'Government Aid', 'None']

In [ ]:
# Triage_Level: A priority rank (by nurses) needs treatment first based on medical urgency.
# Current_Occupancy: The total no. of patients currently inside the ER, representing how "crowded" or busy the hospital is.
# Doctors_on_Shift: The number of medical professionals currently working, which determines the hospital's capacity to see patients.
# Severity_Score: A detailed numerical scale (0–10) that measures exactly how sick or injured an individual patient is.

data={
    'Patient_ID':range(1, n_rows + 1),
    'Age': np.random.randint(1, 95, n_rows),
    'Gender':np.random.choice(genders,n_rows),
    'Arrival_Mode': np.random.choice(arrival_modes, n_rows, p=[0.3, 0.4, 0.05, 0.25]), # p is probability
    'Chief_Complaint': np.random.choice(complaints, n_rows),
    'Insurance_Provider': np.random.choice(insurance, n_rows),
    'Triage_Level': np.random.randint(1, 6, n_rows),
    'Current_Occupancy': np.random.randint(10,100,n_rows),
    'Doctors_on_Shift': np.random.randint(2, 12, n_rows),
    'Severity_Score': np.random.uniform(0, 10, n_rows)
}
df=pd.DataFrame(data)
df

,Patient_ID,Age,Gender,Arrival_Mode,Chief_Complaint,Insurance_Provider,Triage_Level,Current_Occupancy,Doctors_on_Shift,Severity_Score
0,1,52,Non-Binary,Walk-in,Fever,Premium Care,2,75,4,5.420582
1,2,93,Male,Private Vehicle,Fracture,None,3,54,10,4.071208
2,3,15,Male,Ambulance,Fracture,Premium Care,1,23,3,3.313050
3,4,72,Non-Binary,Ambulance,Fracture,None,3,58,9,0.009738
4,5,61,Female,Walk-in,Fracture,Premium Care,4,65,4,8.693568
...,...,...,...,...,...,...,...,...,...,...
19995,19996,92,Female,Helicopter,Abdominal Pain,Standard Health,2,23,11,4.202863
19996,19997,82,Male,Walk-in,Breathing Issues,Premium Care,3,19,6,0.489303
19997,19998,79,Non-Binary,Ambulance,Abdominal Pain,Premium Care,2,65,5,4.702568
19998,19999,22,Female,Walk-in,Chest Pain,Government Aid,3,99,10,4.084602


In [ ]:
df['Wait_Time_Minutes']=30+(df['Current_Occupancy']*1.5)-(df['Doctors_on_Shift']*4)+(df['Triage_Level']*10)

df.loc[df['Arrival_Mode'] == 'Helicopter', 'Wait_Time_Minutes'] -=20
df.loc[df['Arrival_Mode'] == 'Ambulance', 'Wait_Time_Minutes'] -=10

df['Wait_Time_Minutes'] = df['Wait_Time_Minutes'].clip(lower=5).round(1)
df['Wait_Time_Minutes']

,Wait_Time_Minutes
0,146.5
1,101.0
2,52.5
3,101.0
4,151.5
...,...
19995,20.5
19996,64.5
19997,117.5
19998,168.5


In [ ]:
df

,Patient_ID,Age,Gender,Arrival_Mode,Chief_Complaint,Insurance_Provider,Triage_Level,Current_Occupancy,Doctors_on_Shift,Severity_Score,Wait_Time_Minutes
0,1,52,Non-Binary,Walk-in,Fever,Premium Care,2,75,4,5.420582,146.5
1,2,93,Male,Private Vehicle,Fracture,None,3,54,10,4.071208,101.0
2,3,15,Male,Ambulance,Fracture,Premium Care,1,23,3,3.313050,52.5
3,4,72,Non-Binary,Ambulance,Fracture,None,3,58,9,0.009738,101.0
4,5,61,Female,Walk-in,Fracture,Premium Care,4,65,4,8.693568,151.5
...,...,...,...,...,...,...,...,...,...,...,...
19995,19996,92,Female,Helicopter,Abdominal Pain,Standard Health,2,23,11,4.202863,20.5
19996,19997,82,Male,Walk-in,Breathing Issues,Premium Care,3,19,6,0.489303,64.5
19997,19998,79,Non-Binary,Ambulance,Abdominal Pain,Premium Care,2,65,5,4.702568,117.5
19998,19999,22,Female,Walk-in,Chest Pain,Government Aid,3,99,10,4.084602,168.5


In [ ]:
analysis = df.groupby('Triage_Level')['Wait_Time_Minutes'].mean()
print(analysis)

Triage_Level
1     91.580468
2    102.855798
3    112.105005
4    121.351871
5    132.495455
Name: Wait_Time_Minutes, dtype: float64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Patient_ID          20000 non-null  int64  
 1   Age                 20000 non-null  int64  
 2   Gender              20000 non-null  object 
 3   Arrival_Mode        20000 non-null  object 
 4   Chief_Complaint     20000 non-null  object 
 5   Insurance_Provider  20000 non-null  object 
 6   Triage_Level        20000 non-null  int64  
 7   Current_Occupancy   20000 non-null  int64  
 8   Doctors_on_Shift    20000 non-null  int64  
 9   Severity_Score      20000 non-null  float64
 10  Wait_Time_Minutes   20000 non-null  float64
dtypes: float64(2), int64(5), object(4)
memory usage: 1.7+ MB


In [ ]:
# One-Hot Encoding
df_encoded = pd.get_dummies(df,columns = ['Gender', 'Arrival_Mode', 'Chief_Complaint', 'Insurance_Provider'])
X=df_encoded.drop(['Patient_ID','Wait_Time_Minutes'], axis = 1)
Y=df_encoded['Wait_Time_Minutes']

In [ ]:
# The "Train-Test Split" Concept
X_train,X_test,Y_train,Y_test=train_test_split(X,Y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression

# 1. Initialize the model
model = LinearRegression()

# 2. Train the model using the training data
model.fit(X_train, Y_train)

LinearRegression()

In [ ]:
# 3. Make predictions on the test set
y_pred = model.predict(X_test)
# Predict for the entire 20,000 rows
df['Predicted_Wait_Time'] = model.predict(X)

# Calculate the 'Error' (how much the model was off)
df['Prediction_Error'] = df['Wait_Time_Minutes'] - df['Predicted_Wait_Time']

In [ ]:
from sklearn.metrics import r2_score

# 1. Get the R2 Score for the test set
r2 = r2_score(Y_test, y_pred)
print(f"Model R-squared Score: {r2:.4f}")

# 2. Predict for the entire 20,000 rows
df['Predicted_Wait_Time'] = model.predict(X)

# 3. Calculate the 'Error' (Residuals)
df['Prediction_Error'] = df['Wait_Time_Minutes'] - df['Predicted_Wait_Time']

# 4. Save the final file for Power BI
# df.to_csv('Smart_Hospital_ER_Final.csv', index=False)

Model R-squared Score: 1.0000
